# Counting verbs and adjectives

This notebook builds the construal-level score for 8,800 English-language Michelin Guide restaurant introductions and compares groups of restaurants. It is a trial analysis.

**Data** (not in this repo; in my Google Drive folder `restaurant_clt`):
- `michelin_guide_with_meta.csv`: the scraped introductions and restaurant details
- `michelin_analyzed_features.parquet`: the same restaurants with their word counts

**Steps**
1. Add a country column
2. Tag and count verbs and adjectives
3. Score each introduction
4. Compare groups
5. Check group sizes (power analysis)
6. One cuisine across countries

In [ ]:
import os
import sys

# In Colab: connect Google Drive, where the data lives
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.stats.power as smp

# Folder with the data; change this to run the notebook somewhere else
DATA_DIR = os.environ.get("CLT_DATA_DIR", "/content/drive/My Drive/restaurant_clt")
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Add a country column

The country is the last part of each restaurant's address.

In [ ]:
meta = pd.read_csv(f"{DATA_DIR}/michelin_guide_with_meta.csv")
meta["country"] = meta["michelin_address"].str.split(",").str[-1].str.strip()
meta.to_csv(f"{DATA_DIR}/michelin_guide_enriched.csv", index=False)

## 2. Tag and count verbs and adjectives

`michelin_engine.py` tags every word with spaCy (`en_core_web_sm`) and counts the verbs, adjectives and nouns in each introduction. Each word counts once, and helping verbs like *is* are not counted.

Tagging takes a while, so it only runs when `RUN_TAGGING = True`. Otherwise the notebook loads the saved counts.

In [ ]:
RUN_TAGGING = False

if RUN_TAGGING:
    from michelin_engine import extract_clt_features

    counts = extract_clt_features(meta, text_col="michelin_description_26", key_col="url")
    meta.merge(counts, left_on="url", right_on="url_key").to_parquet(
        f"{DATA_DIR}/michelin_analyzed_features.parquet", index=False
    )

df = pd.read_parquet(f"{DATA_DIR}/michelin_analyzed_features.parquet")
print(f"{len(df):,} introductions")

## 3. Score each introduction

**First try: an abstraction index.** The share of adjectives, from 0 (concrete) to 1 (abstract). The 0.001 avoids dividing by zero.

In [ ]:
df["clt_score"] = df["adj_count"] / (df["adj_count"] + df["verb_count"] + 0.001)

plt.figure(figsize=(8, 4))
sns.histplot(df["clt_score"], bins=30, kde=True)
plt.title("Distribution of Abstraction Scores (0=Concrete, 1=Abstract)")
plt.xlabel("Abstraction Index")
plt.show()

# Check the ends: the most concrete should be mostly verbs, the most abstract mostly adjectives
cols = ["url_key", "verb_count", "adj_count", "clt_score"]
print("Most concrete:")
print(df.sort_values("clt_score").head(3)[cols])
print("\nMost abstract:")
print(df.sort_values("clt_score", ascending=False).head(3)[cols])

**Why a polarized score.** With the index, an introduction with 5 verbs and 0 adjectives scores 0, the same as one with nothing to count. A polarized score keeps "only verbs" (−1) apart from "no lean" (0).

In [ ]:
# -1 = only verbs (concrete), 0 = balanced or nothing to count, +1 = only adjectives (abstract)
df["clt_polarity"] = (df["adj_count"] - df["verb_count"]) / (df["adj_count"] + df["verb_count"] + 0.001)

plt.figure(figsize=(10, 5))
sns.histplot(df["clt_polarity"], bins=30, kde=True, color="purple")
plt.axvline(0, color="red", linestyle="--", label="Balanced (0.0)")
plt.title("Distribution of Concrete vs. Abstract Language")
plt.xlabel("← Concrete (Process) ............ Abstract (Essence) →")
plt.legend()
plt.show()

## 4. Compare groups

The polarized score by price level, Michelin distinction and country.

In [ ]:
for group in ["michelin_price_level", "michelin_distinction", "country"]:
    display(df.groupby(group)["clt_polarity"].describe())

In [ ]:
# The 15 countries with the most restaurants, from most concrete to most abstract
top_countries = df["country"].value_counts().nlargest(15).index
countries = df[df["country"].isin(top_countries)]
order = countries.groupby("country")["clt_polarity"].mean().sort_values().index

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(x="country", y="clt_polarity", data=countries, order=order,
            color="lightblue", fliersize=1, linewidth=1, ax=ax)
sns.pointplot(x="country", y="clt_polarity", data=countries, order=order,
              color="darkred", linestyle="none", errorbar=("ci", 95), capsize=0.1, ax=ax)
ax.set_title(f"Abstraction by Country (Top {len(top_countries)})")
ax.set_xlabel("Country (Sorted by Avg Abstraction)")
ax.set_ylabel("← Concrete .......... Abstract →")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.savefig(f"{FIG_DIR}/michelin_descrip_clt_by_region.png", dpi=100, bbox_inches="tight")
plt.show()

Japan is much lower. Is that because foreign words (loanwords) in the introductions are tagged as nouns or verbs and so count as lower, even though their foreignness may make the text more abstract? This is my open check.

In [ ]:
# A restaurant can have several cuisine labels; split them into a list
def parse_categories(cat_str):
    if not isinstance(cat_str, str):
        return []
    delimiter = ";" if ";" in cat_str else ","
    return [c.strip() for c in cat_str.split(delimiter) if c.strip()]


df["cuisine_list"] = df["michelin_categories_list"].apply(parse_categories)

# One row per restaurant and label, so a restaurant with two labels counts under both
cuisine_df = df.explode("cuisine_list")
top_cuisines = cuisine_df["cuisine_list"].value_counts().head(30).index
cuisine_analysis = cuisine_df[cuisine_df["cuisine_list"].isin(top_cuisines)]
print(f"{len(df):,} restaurants -> {len(cuisine_analysis):,} rows for the top {len(top_cuisines)} labels")

In [ ]:
order = cuisine_analysis.groupby("cuisine_list")["clt_polarity"].mean().sort_values(ascending=False).index

plt.figure(figsize=(12, 6))
sns.pointplot(x="cuisine_list", y="clt_polarity", data=cuisine_analysis, order=order,
              color="purple", linestyle="none", capsize=0.1)
plt.title(f"Construal level by Cuisine Type (Top {len(top_cuisines)})")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Cuisine")
plt.ylabel("← Concrete (-1) ........ Abstract (+1) →")
plt.axhline(0, color="grey", linestyle="--", alpha=0.5)
plt.savefig(f"{FIG_DIR}/cl_by_cuisine.png", dpi=100, bbox_inches="tight")
plt.show()

## 5. Check group sizes (power analysis)

The groups are very different in size, as expected. A power analysis gives the smallest group size needed to detect a medium effect.

In [ ]:
# Each restaurant's first label is its primary cuisine
df["primary_cuisine"] = df["cuisine_list"].apply(lambda labels: labels[0] if labels else "Unknown")

# Keep primary cuisines with at least 50 restaurants
threshold = 50
cuisine_counts = df["primary_cuisine"].value_counts()
subset_primary = df[df["primary_cuisine"].isin(cuisine_counts[cuisine_counts >= threshold].index)]

print(f"{len(df):,} restaurants -> {len(subset_primary):,} in cuisines with at least {threshold}")
print(subset_primary["primary_cuisine"].value_counts().head(5))

In [ ]:
# Cohen's f: 0.10 small, 0.25 medium, 0.40 large
effect_size = 0.25
alpha = 0.05
power = 0.80
k_groups = 10  # number of groups compared, e.g. the top 10 cuisines

required_n = smp.FTestAnovaPower().solve_power(
    effect_size=effect_size, nobs=None, alpha=alpha, power=power, k_groups=k_groups
)
min_per_group = int(np.ceil(required_n / k_groups))

print(f"Total N needed: {int(required_n)}")
print(f"Minimum N per group: {min_per_group}")

counts = subset_primary["primary_cuisine"].value_counts()
print(f"\nGroups with at least {min_per_group}:", counts[counts >= min_per_group].index.tolist())
print(f"Groups with fewer (leave out):", counts[counts < min_per_group].index.tolist())

## 6. One cuisine across countries

Japanese restaurants (primary cuisine) in the countries with more than 10 of them.

In [ ]:
target_cuisine = "Japanese"
japanese = subset_primary[subset_primary["primary_cuisine"] == target_cuisine]
country_counts = japanese["country"].value_counts()
valid_countries = country_counts[country_counts > 10].index
japanese = japanese[japanese["country"].isin(valid_countries)]
print(f"{target_cuisine} food in: {list(valid_countries)}")

order = japanese.groupby("country")["clt_polarity"].mean().sort_values().index

plt.figure(figsize=(10, 6))
sns.pointplot(x="country", y="clt_polarity", data=japanese, order=order,
              color="darkred", linestyle="none", capsize=0.1)
plt.axhline(0, color="grey", linestyle="--")
plt.title(f"How '{target_cuisine}' Food is Described in Different Countries")
plt.ylabel("← Concrete Process .......... Abstract Essence →")
plt.savefig(f"{FIG_DIR}/j_different_region.png", dpi=100, bbox_inches="tight")
plt.show()

## Notes for later

- A finer measure could separate the kinds of verbs (descriptive action verbs, interpretive action verbs and state verbs), as the Linguistic Category Model does.
- Another option is concreteness ratings for 40,000 English words (Brysbaert, Warriner & Kuperman, 2014). They rate nouns too ("fish" is concrete, "cuisine" is abstract), so they still work when an introduction has no adjectives.